<a href="https://colab.research.google.com/github/1900690/cucumber/blob/main/cucumber_counrter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#zipを解凍
import shutil
import os

file_name ="cucanber-fruit-flower_20260409142047.zip"

#データを解凍
shutil.unpack_archive('/content/'+file_name, '/content/')
#zipを消す
os.remove('/content/'+file_name)

In [ ]:
# @title 📦 物体検出用（YOLO）データセット構築 & YAML作成
import os
import yaml
import shutil
import cv2
import numpy as np
import albumentations as A
from sklearn.model_selection import train_test_split
from tqdm.notebook import tqdm

# --- 基本設定 ---
# @markdown ### パスと分割の設定
source_img_dir = '/content/original' # @param {type:"string"}
source_label_dir = '/content/yolo/annotations' # @param {type:"string"}
classes_file = '/content/yolo/classes.txt' # @param {type:"string"}
output_root_dir = '/content/dataset/detect' # @param {type:"string"}

# @markdown ---
# @markdown ### 分割比率
train_ratio = 0.7 # @param {type:"slider", min:0, max:1, step:0.1}
val_ratio = 0.2 # @param {type:"slider", min:0, max:1, step:0.1}
test_ratio = 0.1 # @param {type:"slider", min:0, max:1, step:0.1}

# 既存フォルダの初期化
shutil.rmtree(output_root_dir, ignore_errors=True)

# --- オーグメンテーション設定 ---
apply_to_train = True
apply_to_val = False
apply_to_test = False
num_augmented_per_image = 2

def get_augmentation_pipeline():
    return A.Compose([
        A.Rotate(limit=30, p=0.5),
        A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.1, rotate_limit=0, p=0.5),
        A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
        A.GaussNoise(var_limit=(10.0, 50.0), p=0.3),
        A.Blur(blur_limit=3, p=0.3)
    ], bbox_params=A.BboxParams(format='yolo', label_fields=['class_labels']))

def load_yolo_labels(path):
    bboxes, class_labels = [], []
    if os.path.exists(path):
        with open(path, 'r') as f:
            for line in f.readlines():
                parts = list(map(float, line.split()))
                class_labels.append(int(parts[0]))
                bboxes.append(parts[1:])
    return bboxes, class_labels

def save_yolo_labels(path, bboxes, class_labels):
    with open(path, 'w') as f:
        for label, bbox in zip(class_labels, bboxes):
            f.write(f"{label} {' '.join([f'{x:.6f}' for x in bbox])}\n")

def process_and_setup_yolo():
    # クラス名の読み込み
    with open(classes_file, 'r') as f:
        class_names = [line.strip() for line in f.readlines() if line.strip()]

    # 画像リストの取得
    img_extensions = ('.jpg', '.jpeg', '.png', '.JPG')
    all_images = [f for f in os.listdir(source_img_dir) if f.endswith(img_extensions)]

    # 1. データ分割
    train_files, temp_files = train_test_split(all_images, test_size=(1 - train_ratio), random_state=42)
    rel_val_ratio = val_ratio / (val_ratio + test_ratio) if (val_ratio + test_ratio) > 0 else 0
    val_files, test_files = train_test_split(temp_files, test_size=(1 - rel_val_ratio), random_state=42) if len(temp_files) > 1 else (temp_files, [])

    split_dict = {'train': train_files, 'val': val_files, 'test': test_files}
    aug_settings = {'train': apply_to_train, 'val': apply_to_val, 'test': apply_to_test}
    aug_pipeline = get_augmentation_pipeline()

    # 2. フォルダ作成
    for split in ['train', 'val', 'test']:
        os.makedirs(os.path.join(output_root_dir, 'images', split), exist_ok=True)
        os.makedirs(os.path.join(output_root_dir, 'labels', split), exist_ok=True)

    # 3. 処理実行
    print("--- データの整理と拡張を開始 ---")
    for split, files in split_dict.items():
        if not files: continue
        for filename in tqdm(files, desc=f"Processing {split}"):
            basename = os.path.splitext(filename)[0]
            img_path = os.path.join(source_img_dir, filename)
            label_path = os.path.join(source_label_dir, basename + '.txt')

            image = cv2.imread(img_path)
            if image is None: continue
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            bboxes, class_labels = load_yolo_labels(label_path)

            # オリジナル保存
            cv2.imwrite(os.path.join(output_root_dir, 'images', split, filename), cv2.cvtColor(image, cv2.COLOR_RGB2BGR))
            if os.path.exists(label_path):
                shutil.copy2(label_path, os.path.join(output_root_dir, 'labels', split, basename + '.txt'))

            # オーグメンテーション
            if aug_settings[split]:
                for i in range(num_augmented_per_image):
                    try:
                        augmented = aug_pipeline(image=image, bboxes=bboxes, class_labels=class_labels)
                        if len(augmented['bboxes']) > 0:
                            aug_name = f"{basename}_aug_{i}"
                            cv2.imwrite(os.path.join(output_root_dir, 'images', split, f"{aug_name}.jpg"), cv2.cvtColor(augmented['image'], cv2.COLOR_RGB2BGR))
                            save_yolo_labels(os.path.join(output_root_dir, 'labels', split, f"{aug_name}.txt"), augmented['bboxes'], class_labels)
                    except: continue

    # 4. detect.yaml の作成
    yaml_data = {
        'train': 'images/train',
        'val': 'images/val',
        'test': 'images/test',
        'names': {i: name for i, name in enumerate(class_names)}
    }

    yaml_path = os.path.join(output_root_dir, 'detect.yaml')
    with open(yaml_path, 'w') as f:
        yaml.dump(yaml_data, f, default_flow_style=False, sort_keys=False)

    print(f"\n✅ 完了！")
    print(f"データセット場所: {output_root_dir}")
    print(f"作成されたYAML: {yaml_path}")

# 実行
process_and_setup_yolo()